# Challenge 6: Feedback and annotation queues

**EDD: Measure. Model upgrade framework, step 4 (trust), first half: put humans where they matter.** Reviewing every trace doesn't scale, and a judge can only be trusted against human labels.
You'll capture feedback with `client.create_feedback`, write the rules that decide which runs a person should look at, and fill an annotation queue with exactly those runs.
Challenge 7 then uses reviewed labels to calibrate an LLM judge.

In [ ]:
from pathlib import Path
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'pyproject.toml').exists())

import json
import os
import sys
import warnings


for path in (str(ROOT), str(ROOT / "helpers")):
    if path not in sys.path:
        sys.path.insert(0, path)

from dotenv import load_dotenv
load_dotenv(ROOT / ".env", override=True)

from langsmith import Client, uuid7

import lab_links
import lab_sdk

client = Client()
PROD = lab_sdk.prod_project()
GOLDEN = lab_sdk.ds_golden()
print("prod project:", PROD)
print("golden dataset:", GOLDEN)

from collections import Counter
from concurrent.futures import ThreadPoolExecutor

from eval_lab.evaluators import judge_json
QUEUE = lab_sdk.queue_name()

## 1. Capture feedback

`client.create_feedback(run_id, key, session_id=..., score=..., comment=..., correction=...)` attaches feedback to a run.
`correction` holds what the answer should have been, which later becomes a dataset example.

`submit_feedback` below is a thin wrapper that uses the key `user_rating`.

In [ ]:
def submit_feedback(run_id, rating, comment=None, correction=None, session_id=None):
    return client.create_feedback(
        run_id=run_id, session_id=session_id, key="user_rating", score=rating, comment=comment,
        correction=correction, source_info={"channel": "sdk"},
    )

Now use it. Users rate by **outcome**: a refund that comes through is a 1, a refund that is denied is a 0. That is
not the same as "the agent followed the process", which is why the two signals will disagree.

In [ ]:
all_roots = lab_sdk.roots(client)
refund_runs = [r for r in all_roots if "refund" in (r.tags or [])]

for r in refund_runs:
    got_refund = "issue_refund" in (r.outputs or {}).get("trajectory", [])
    submit_feedback(
        r.id,
        session_id=r.session_id,
        rating=int(got_refund),
        comment="Refund came through." if got_refund else "Refund was denied.",
        correction=None if got_refund else {"outputs": {"answer": "Explain the 30-day policy and offer an alternative."}},
    )

# auto_score: our own process check, stored beside the user's rating.
for r in all_roots:
    client.create_feedback(run_id=r.id, session_id=r.session_id, key="auto_score", score=lab_sdk.auto_score(r),
                           source_info={"evaluator": "auto_score@v1"})
client.flush()

ids = [r.id for r in refund_runs]
seen = lab_sdk.wait_for_feedback(client, "user_rating", ids, expected=len(ids))
print(f"user_rating on {len(seen)} of {len(ids)} refund runs")

## 2. Review rules

Reviewing everything doesn't scale, so write rules. `rows` has one dict per run:
`run_id`, `auto_score` (our process check), `user_rating` (or `None`) and `error`.

Write `select_for_review(rows)`. It returns `{run_id: [reasons]}` and picks a run when:

- `auto_score` is 0 (`low_score`)
- `auto_score` and `user_rating` differ by 1 or more (`disagreement`)
- the run errored (`error`)
- a deterministic one-in-five sample, using the run id's last two hex digits (`sample`)

A reference solution is in the next cell.

In [ ]:
ratings = {f.run_id: f.score for f in client.list_feedback(run_ids=[r.id for r in all_roots], feedback_key=["user_rating"])}
rows = [{"run_id": r.id, "auto_score": lab_sdk.auto_score(r), "user_rating": ratings.get(r.id), "error": bool(r.error)}
        for r in all_roots]
print(len(rows), "rows;", sum(1 for r in rows if r["user_rating"] is not None), "with a user rating")

In [ ]:
# Reference solution: try writing this yourself first.
def select_for_review(rows):
    picked = {}
    for row in rows:
        reasons = []
        if row["auto_score"] == 0:
            reasons.append("low_score")
        if row["user_rating"] is not None and abs(row["auto_score"] - row["user_rating"]) >= 1:
            reasons.append("disagreement")
        if row["error"]:
            reasons.append("error")
        if int(str(row["run_id"]).replace("-", "")[-2:], 16) % 5 == 0:
            reasons.append("sample")
        if reasons:
            picked[row["run_id"]] = reasons
    return picked

In [ ]:
picked = select_for_review(rows)
assert picked, "select_for_review returned nothing. Check your rules against the rows above."
from lab_state import mark_done
mark_done("/tmp/.sdk-ch6-queue-ok")

print(len(picked), "of", len(rows), "runs selected")
print(dict(Counter(reason for reasons in picked.values() for reason in reasons)))

## 3. Fill the queue

`create_annotation_queue` makes the queue; `add_runs_to_annotation_queue` fills it. Reviewers then label runs in the LangSmith UI
(**Annotation Queues**). The cell is idempotent: it only adds runs the queue doesn't already hold. Annotation queues need a LangSmith
plan that includes them.

In [ ]:
queue = next(iter(client.list_annotation_queues(name=QUEUE)), None) or client.create_annotation_queue(
    name=QUEUE, description="Disagreements, failures and a random sample")
existing = {r.id for r in client.list_runs_from_annotation_queue(queue.id)}
new = [rid for rid in picked if rid not in existing]
if new:
    by_id = {r.id: r for r in all_roots}
    client.add_runs_to_annotation_queue(queue.id, runs=[
        {"run_id": rid, "session_id": by_id[rid].session_id, "start_time": by_id[rid].start_time} for rid in new])
print(f"queue {QUEUE}: added {len(new)} runs")

import time
time.sleep(3)
print("queue now holds", len(list(client.list_runs_from_annotation_queue(queue.id))), "runs")

## 4. See it in LangSmith

In [ ]:
lab_links.queue_link(QUEUE, hint="what a reviewer sees; add feedback, and use Add to dataset on good runs")
lab_links.project_link(PROD, hint="filter runs with feedback user_rating to see what the rules were given")